# AGAR-RL V10 : MaskablePPO SOTA - Profil NVIDIA A100

[![Open In Colab](https://colab.research.google.com/github/Albin0903/agario/blob/main/notebooks/train_colab.ipynb)](https://colab.research.google.com/github/Albin0903/agario/blob/main/notebooks/train_colab.ipynb)

Pipeline V10 optimise pour A100 40 Go: profil `n_envs/batch_size` sélectionné par benchmark, rollout `2048`, 10 epoques PPO, 20 slots de ligue, LayerNorm/SiLU, cosine annealing et self-play PFSP.

### Objectif de reward V10
1. Toute croissance nette de masse est recompensee, meme sous le **episode peak mass**.
2. Un bonus supplementaire est accorde lorsqu'un nouveau peak est etabli.
3. Toute perte de masse ou de fragments est penalisee au moment ou elle survient.
4. La mort ajoute une penalite terminale bornee, sans micro-penalites de bord, de virage ou de split.
5. `MaskablePPO` interdit les splits physiquement invalides: masse inferieure a 36 ou 16 sous-cellules.
6. Le learning rate descend de `3e-4` a `1e-5`; l'entropie descend de `0.005` a `0.001`.
7. TF32/Tensor Cores sont activés sur CUDA; la physique reste parallele sur CPU.
8. Si le checkpoint V9 est un PPO vanilla incompatible avec V10, il est utilise comme **teacher** de distillation BC; ses comportements sont conserves avant l'optimisation MaskablePPO.
9. Les anciennes versions servent uniquement de fallback de reprise.

# 1. Montage sécurisé de Google Drive, Détection Ressources A100 & CPU Threads
import os, sys, time, psutil, torch

# 1. Empêcher la sur-allocation de threads CPU (OpenMP contention)
torch.set_num_threads(1)

try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
except ImportError:
    pass

DRIVE_BACKUP_DIR = '/content/drive/MyDrive/agario_rl_backup_v10'
PREV_BACKUP_V9 = '/content/drive/MyDrive/agario_rl_backup_v9'
PREV_BACKUP_V8 = '/content/drive/MyDrive/agario_rl_backup_v8'
PREV_BACKUP_V7 = '/content/drive/MyDrive/agario_rl_backup_v7'
PREV_BACKUP_V6 = '/content/drive/MyDrive/agario_rl_backup_v6'
PREV_BACKUP_V5 = '/content/drive/MyDrive/agario_rl_backup_v5'
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)

print('=== RESSOURCES SYSTEME COLAB ===')
print(f'Cœurs logiques (vCPUs) : {os.cpu_count()}')
print(f'Cœurs physiques        : {psutil.cpu_count(logical=False)}')
print(f'RAM totale             : {psutil.virtual_memory().total / (1024**3):.2f} Go')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    torch.set_float32_matmul_precision('high')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print(f'GPU disponible         : {gpu_name} ({vram:.1f} Go VRAM)')
else:
    print('GPU disponible         : Aucun')
print(f'Threads PyTorch CPU    : {torch.get_num_threads()}')
print(f'Backup V10             : {DRIVE_BACKUP_DIR}')


In [ ]:
# 1. Montage sécurisé de Google Drive et configuration A100
import os, sys, time, torch

try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
except ImportError:
    pass

DRIVE_BACKUP_DIR = '/content/drive/MyDrive/agario_rl_backup_v10'
PREV_BACKUP_V9 = '/content/drive/MyDrive/agario_rl_backup_v9'
PREV_BACKUP_V8 = '/content/drive/MyDrive/agario_rl_backup_v8'
PREV_BACKUP_V7 = '/content/drive/MyDrive/agario_rl_backup_v7'
PREV_BACKUP_V6 = '/content/drive/MyDrive/agario_rl_backup_v6'
PREV_BACKUP_V5 = '/content/drive/MyDrive/agario_rl_backup_v5'
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    torch.set_float32_matmul_precision('high')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print(f'GPU détecté : {gpu_name} ({vram:.1f} Go VRAM)')
    if 'A100' not in gpu_name.upper():
        print('Avertissement : ce notebook utilise le profil A100 sur un autre GPU.')
else:
    print('Aucun GPU détecté.')
print(f'Backup V10 : {DRIVE_BACKUP_DIR}')
print('Profil : 24 environnements, batch 2048, 10 epochs, Tensor Cores/TF32')

## 1. Synchronisation du Code GitHub & Installation des Dépendances

In [ ]:
import os

# 1. Récupération propre des dernières modifications ou clonage
if os.path.exists('.git'):
    print('🔄 Synchronisation avec GitHub main...')
    !git fetch origin main
    !git reset --hard origin/main
elif os.path.exists('agario/.git'):
    print('🔄 Déplacement dans agario et synchronisation avec GitHub main...')
    %cd agario
    !git fetch origin main
    !git reset --hard origin/main
else:
    print('🌐 Clonage propre du repo...')
    !git clone https://github.com/Albin0903/agario.git
    %cd agario

# 2. Configuration du PYTHONPATH et installation des dépendances Farama Gymnasium
os.environ['PYTHONPATH'] = f"{os.getcwd()}:{os.environ.get('PYTHONPATH', '')}"
!pip uninstall -y -q gym 2>/dev/null || true
!pip install -q -r requirements.txt tensorboard
!apt-get install -qq -y ffmpeg
print('✅ Environnement et dépendances installés avec succès.')

## 2. Validation Pré-Vol : Suite Complète de 34 Tests Unitaires
Vérification complète de la physique du moteur, du remerge magnétique, des récompenses de traque et de l'espace d'observation log-ratio.

In [ ]:
# Validation physique, Gymnasium, reward V10, masking, self-play et export
!python -m pytest -q

## 4. Entraînement Haute Performance V10 (MaskablePPO + LayerNorm + Action Masking)
- **Action Masking Strict** : Empêche structurellement l'agent de tenter des splits illégaux.
- **Sauvegarde Continue V10** : Checkpoints automatiques tous les 250 000 pas dans `agario_rl_backup_v10`.

In [ ]:
# Batch benchmark A100: sélectionne automatiquement le meilleur profil CPU/GPU
import os, sys, time, shutil, tempfile, subprocess, re


def run_profile(n_envs, batch_size, total_steps=12000):
    tag = f'envs_{n_envs}_batch_{batch_size}'
    root = os.path.join(tempfile.gettempdir(), f'agario_v10_probe_{tag}')
    shutil.rmtree(root, ignore_errors=True)
    command = [
        sys.executable, 'src/training/train_colab.py',
        '--n-envs', str(n_envs),
        '--batch-size', str(batch_size),
        '--n-steps', '2048',
        '--total-timesteps', str(total_steps),
        '--no-warm-start',
        '--fresh',
        '--save-dir', os.path.join(root, 'ppo'),
        '--history-dir', os.path.join(root, 'pool'),
        '--device', 'cuda',
    ]
    started = time.perf_counter()
    process = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    elapsed = time.perf_counter() - started
    fps_match = re.search(r"\|\s+fps\s+\|\s+(\d+)\s+\|", process.stdout)
    if fps_match:
        steps_per_second = float(fps_match.group(1))
    else:
        steps_per_second = (n_envs * 2048) / max(elapsed, 1e-6)
    print(f'envs={n_envs:2d} | batch={batch_size:4d} | {steps_per_second:7.1f} steps/s (real SB3 FPS) | {elapsed:6.1f}s | exit={process.returncode}')
    if process.returncode != 0:
        print(process.stdout[-2000:])
    return {
        'n_envs': n_envs,
        'batch_size': batch_size,
        'steps_per_second': steps_per_second,
        'elapsed': elapsed,
        'exit': process.returncode,
    }


# Profils courts compatibles avec n_steps=2048 et les Tensor Cores A100.
CANDIDATE_PROFILES = [(8, 1024), (12, 2048), (16, 2048), (24, 2048)]
profile_results = [run_profile(n_envs, batch_size) for n_envs, batch_size in CANDIDATE_PROFILES]
valid_profiles = [result for result in profile_results if result['exit'] == 0]
BEST_PROFILE = max(valid_profiles, key=lambda result: result['steps_per_second']) if valid_profiles else {
    'n_envs': 24,
    'batch_size': 2048,
    'steps_per_second': 0.0,
}
BEST_N_ENVS = BEST_PROFILE['n_envs']
BEST_BATCH_SIZE = BEST_PROFILE['batch_size']
print(f"\nProfil A100 retenu: n_envs={BEST_N_ENVS}, batch_size={BEST_BATCH_SIZE}, throughput={BEST_PROFILE['steps_per_second']:.1f} steps/s")

In [ ]:
# Profilage des phases physiques: mesure courte, sans modifier les checkpoints
import time
import numpy as np
from src.env.agar_engine import AgarEngine

engine = AgarEngine(width=2000.0, height=2000.0, num_pellets=500, num_viruses=10, seed=42)
for player_id in range(11):
    engine.spawn_player(player_id, initial_mass=20.0)
actions = {player_id: np.array([1.0, 0.0, -1.0], dtype=np.float32) for player_id in range(11)}
for _ in range(200):
    engine.step(actions)

engine.set_profiling(True)
phase_totals = {}
profile_steps = 1000
started = time.perf_counter()
for _ in range(profile_steps):
    engine.step(actions)
    for name, duration in engine.last_profile.items():
        phase_totals[name] = phase_totals.get(name, 0.0) + duration
elapsed = time.perf_counter() - started

print(f'Engine profiling FPS: {profile_steps / elapsed:.1f}')
for name, duration in sorted(phase_totals.items(), key=lambda item: item[1], reverse=True):
    print(f'{name:18s}: {100.0 * duration / elapsed:6.2f}%')

In [ ]:
# Configuration et lancement du profil A100 V10 sélectionné par le benchmark
import os, glob, re

V10_DIR = '/content/drive/MyDrive/agario_rl_backup_v10'
PREVIOUS_DIRS = [
    '/content/drive/MyDrive/agario_rl_backup_v9',
    '/content/drive/MyDrive/agario_rl_backup_v8',
    '/content/drive/MyDrive/agario_rl_backup_v7',
    '/content/drive/MyDrive/agario_rl_backup_v6',
    '/content/drive/MyDrive/agario_rl_backup_v5',
]
os.makedirs(V10_DIR, exist_ok=True)

# Fallback si le batch benchmark n'a pas ete execute.
TRAIN_N_ENVS = int(globals().get('BEST_N_ENVS', 8))
TRAIN_BATCH_SIZE = int(globals().get('BEST_BATCH_SIZE', 1024))

def extract_step(path):
    if 'final' in os.path.basename(path):
        return 999_999_999
    match = re.search(r'step_(\d+)', os.path.basename(path))
    return int(match.group(1)) if match else 0

def find_best_checkpoint(directory):
    if not os.path.exists(directory):
        return None
    candidates = [
        path for path in glob.glob(os.path.join(directory, '*.zip'))
        if os.path.getsize(path) > 1000
        and not os.path.basename(path).startswith('._')
        and 'bc_pretrained' not in os.path.basename(path)
    ]
    return max(candidates, key=extract_step) if candidates else None

chosen_checkpoint = find_best_checkpoint(V10_DIR)
if chosen_checkpoint is None:
    for previous_dir in PREVIOUS_DIRS:
        chosen_checkpoint = find_best_checkpoint(previous_dir)
        if chosen_checkpoint:
            break

resume_flag = f'--resume "{chosen_checkpoint}"' if chosen_checkpoint else '--resume auto'
print(f'Reprise V10 : {chosen_checkpoint or "nouvel entrainement"}')
print(f'Profil A100 retenu : {TRAIN_N_ENVS} envs, n_steps=2048, batch_size={TRAIN_BATCH_SIZE}, n_epochs=10')

!python src/training/train_colab.py \
    --n-envs {TRAIN_N_ENVS} \
    --max-rivals 2 \
    --total-timesteps 20000000 \
    --batch-size {TRAIN_BATCH_SIZE} \
    --n-steps 2048 \
    --pool-interval 250000 \
    --backup-dir {V10_DIR} \
    {resume_flag} \
    --device cuda

## 5. Inspection Diagnostique de la Politique & Réflexes Tactiques
Sonde le réseau de neurones sur des scénarios synthétiques contrôlés : réponse à la nourriture, esquive des prédateurs mortels vs calme face aux rivaux inoffensifs, et propension à attaquer les proies en zone de frappe.

In [ ]:
import os, glob, re
from IPython.display import HTML, display
from base64 import b64encode

def extract_step(path):
    if 'final' in os.path.basename(path):
        return 999999999
    m = re.search(r'step_(\d+)', path)
    return int(m.group(1)) if m else 0

def get_best_in_dir(dir_path):
    if not os.path.exists(dir_path):
        return None
    zips = glob.glob(os.path.join(dir_path, '*.zip'))
    valid = [z for z in zips if os.path.getsize(z) > 1000 and not os.path.basename(z).startswith('._') and 'bc_pretrained' not in z]
    if not valid:
        return None
    valid.sort(key=extract_step, reverse=True)
    return valid[0]

search_dirs = [
    '/content/drive/MyDrive/agario_rl_backup_v10',
    '/content/drive/MyDrive/agario_rl_backup_v9',
    '/content/drive/MyDrive/agario_rl_backup_v8',
    '/content/drive/MyDrive/agario_rl_backup_v7',
    '/content/drive/MyDrive/agario_rl_backup_v6',
    'checkpoints/ppo'
]

target_model = None
for sd in search_dirs:
    target_model = get_best_in_dir(sd)
    if target_model:
        break
target_model = target_model or 'checkpoints/ppo/ppo_latest.zip'
step_count = extract_step(target_model)

print('=' * 75)
print(f'🎬 Modèle sélectionné pour le Replay HD : {target_model}')
print(f'📊 Palier : {step_count:,} steps' if step_count < 999_999_999 else '📊 Palier : FINAL')
print('=' * 75)

os.makedirs('recordings', exist_ok=True)
!python src/inference/record_match.py \
    --model "{target_model}" \
    --output recordings/eval_match_v10.mp4 \
    --steps 2400

if os.path.exists('recordings/eval_match_v10.mp4') and os.path.exists('/content/drive/MyDrive/agario_rl_backup_v10'):
    !cp recordings/eval_match_v10.mp4 /content/drive/MyDrive/agario_rl_backup_v10/eval_match_v10.mp4
    print('📁 Replay HD copié sur Google Drive dans : agario_rl_backup_v10/eval_match_v10.mp4')

video_path = 'recordings/eval_match_v10.mp4'
if os.path.exists(video_path):
    mp4_bytes = open(video_path, 'rb').read()
    data_url = 'data:video/mp4;base64,' + b64encode(mp4_bytes).decode()
    display(HTML(f'''
    <video width="850" height="480" controls autoplay loop>
        <source src="{data_url}" type="video/mp4">
    </video>
    '''))
    print(f'Taille de la vidéo : {os.path.getsize(video_path) / 1_000_000:.1f} Mo')
else:
    print('⚠️ Vidéo non trouvée.')
